# Infer — Mask R-CNN `best.pt` на всём val + submit

Не учим. GPU T4, Internet on.
Add Data: competition `filament-segmentation-2026` **и** Output прогона `dm1triyaleksandrov/notebookead6e030d5`.
Secrets: `GITHUB_TOKEN` на этот ноутбук.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Settings → GPU T4, Restart session"

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os

token = ""
try:
    sec = UserSecretsClient()
    for name in ("GITHUB_TOKEN", "GH_TOKEN", "GIT_TOKEN", "github_token"):
        try:
            value = sec.get_secret(name)
        except Exception:
            value = ""
        if value:
            token = value
            break
except Exception as e:
    print("secrets:", type(e).__name__, e)

os.environ["GITHUB_TOKEN"] = token
if not token:
    raise SystemExit("Add-ons → Secrets → GITHUB_TOKEN на ЭТОТ ноутбук, потом Restart")

askpass = """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
Path("/kaggle/working/git-askpass.sh").write_text(askpass)
os.chmod("/kaggle/working/git-askpass.sh", 0o700)

REPO = Path("/kaggle/working/repo")
if not (REPO / "run.py").exists():
    !GIT_ASKPASS=/kaggle/working/git-askpass.sh GIT_TERMINAL_PROMPT=0 \
      git clone --branch feature/mask-rcnn --single-branch \
      https://github.com/d1maal/filament-segmentation-2026.git /kaggle/working/repo

%cd /kaggle/working/repo
!git log -1 --oneline
%pip install -q pycocotools python-dotenv tqdm

In [ ]:
import csv
import json
from pathlib import Path

import cv2
import numpy as np
import torch
from tqdm import tqdm

from src.config import config
from src.data import (
    ensure_val_split,
    image_stem,
    instances_for_image,
    load_coco,
    mask_to_rle_counts,
    resolve_image,
    three_channel,
)
from src.metrics import aggregate, panoptic_quality
from src.models.mask_rcnn.infer import load_checkpoint, predict_masks

WEIGHTS = Path("/kaggle/working/outputs/mask_rcnn/best.pt")
if not WEIGHTS.exists():
    hits = sorted(Path("/kaggle/input").rglob("best.pt"))
    assert hits, "Add Data → Output прогона с best.pt"
    WEIGHTS = hits[0]

SCORE_THR = 0.30
MIN_AREA = 80
MIN_PQ_FOR_SUBMIT = 0.20
OUT = Path("/kaggle/working/outputs/mask_rcnn")
OUT.mkdir(parents=True, exist_ok=True)
print("weights:", WEIGHTS)
print("dataset:", config.dataset_path, "images:", config.train_images.exists())


def val_items():
    coco = load_coco()
    val = ensure_val_split(coco)
    items = []
    for info in coco.imgs.values():
        if image_stem(info["file_name"]) not in val:
            continue
        path = resolve_image(config.train_images, info["file_name"])
        if path is not None:
            items.append((coco, info, path))
    items.sort(key=lambda item: image_stem(item[1]["file_name"]))
    return items


def to_tensor(gray: np.ndarray) -> torch.Tensor:
    rgb = three_channel(gray)
    return torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float() / 255.0


model, device = load_checkpoint(WEIGHTS)
items = val_items()
rows = []
print(f"full val: {len(items)} images | {WEIGHTS.name} | score={SCORE_THR}", flush=True)
for coco, info, path in tqdm(items, desc="val"):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise SystemExit(f"Failed to read {path}")
    pred = predict_masks(
        model, to_tensor(gray), device,
        score_threshold=SCORE_THR, min_area=MIN_AREA,
    )
    metrics = panoptic_quality(instances_for_image(coco, info["id"]), pred)
    metrics["stem"] = Path(info["file_name"]).stem
    rows.append(metrics)

summary = aggregate(rows)
print("Home PQ full val:", round(summary["pq"], 4), summary)
SUBMIT = summary["pq"] >= MIN_PQ_FOR_SUBMIT
print("submit:", SUBMIT, f"(threshold {MIN_PQ_FOR_SUBMIT})")
(OUT / "val_metrics.json").write_text(
    json.dumps({"weights": str(WEIGHTS), "summary": summary, "per_image": rows}, indent=2),
    encoding="utf-8",
)

if SUBMIT:
    csv_rows = []
    test_images = sorted(
        p for p in config.test_images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    print(f"test: {len(test_images)}", flush=True)
    for path in tqdm(test_images, desc="test"):
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise SystemExit(f"Failed to read {path}")
        pred = predict_masks(
            model, to_tensor(gray), device,
            score_threshold=SCORE_THR, min_area=MIN_AREA,
        )
        for i, mask in enumerate(pred, start=1):
            csv_rows.append(
                {"filament_id": f"{path.stem}_{i}", "segmentation_rle": mask_to_rle_counts(mask)}
            )
    sub = OUT / "submission_mask_rcnn.csv"
    with sub.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, ["filament_id", "segmentation_rle"])
        w.writeheader()
        w.writerows(csv_rows)
    print(f"Wrote {sub} ({len(csv_rows)} rows)")
    !ls -lh /kaggle/working/outputs/mask_rcnn/submission_mask_rcnn.csv /kaggle/working/outputs/mask_rcnn/val_metrics.json